# ThreeR3C_TiThTe — VUBC

3R3C (states $T_i$, $T_h$, $T_e$), temperature task: $T_i$ and $T_h$ supervised.

Prepares `data_train.joblib` and `identification_config.json`; the model is trained with `rct-train` (last cell). Data split as in `six_r3c_eniso13790/VUBC`: season w2324, training windows of 3 h sliding by 1 h from 1 Dec 2023 to 1 Feb 2024, 24 h pre-window lag, no validation split.

In [1]:
import json
import os
from datetime import datetime, timedelta

import joblib
import pandas as pd

from rctoolbox.utils.project_root import find_project_root
from rctoolbox.data.time_windows_generator import generate_time_windows
from rctoolbox.data.numpydataloader import DataLoader as NumpyDataLoader

project_root = find_project_root()
os.chdir(project_root)

In [2]:
HEMS_ID = "VUBC"
OUT_DIR_PATH = project_root / "examples" / "run_identification" / "ThreeR3C_TiThTe" / HEMS_ID

## DATA

In [3]:
DATA_DIR = project_root / "examples" / "data"
DATA = pd.read_pickle(DATA_DIR / f"{HEMS_ID}_data.pkl")
SEASON = "w2324"
df = DATA[SEASON]["df"]

print("season:        ", SEASON)
print("df shape:      ", df.shape)
print("time range:    ", df.index.min(), "->", df.index.max())

season:         w2324
df shape:       (3721, 10)
time range:     2023-10-30 00:00:00 -> 2024-04-02 00:00:00


### TRAIN SPLIT

In [4]:
wdh = 3; spb = int(1e10); ssh = 1
pre_window_lag = timedelta(hours=24)

t1_train = datetime(2023, 12, 1)
t2_train = datetime(2024, 2, 1)
tws_train = generate_time_windows(start_time=t1_train, end_time=t2_train, window_duration_hours=wdh, samples_per_batch=spb, slide_step_hours=ssh)

tws = {"train": tws_train}
dataloader = NumpyDataLoader(df)
dataloader.load_all_data_batches(time_windows=tws, features=df.columns.tolist(), pre_window_lag=pre_window_lag)

In [5]:
from rctoolbox.models.rc.ThreeR3C_TiThTe.build_data_dict import build_data_dict
COLUMNS = {"T_h": "y2"}   # model signal -> data column (T_i = y1, T_a, Phi_sol, Phi_HC by default)
build = lambda split: build_data_dict(dataloader, split, columns=COLUMNS)

data_joblib = {"train": build("train"), "val": None, "columns": COLUMNS}
data_joblib_path = OUT_DIR_PATH / "data_train.joblib"
joblib.dump(data_joblib, data_joblib_path)
print("train windows:", data_joblib["train"]["t"].shape)

train windows: (4,)


## IDENTIFICATION CONFIG

In [6]:
model_config = {
    "model_type":    "3R3C_temperature",
    "init_config":   {"method": "dynamic", "lag_h": 24},   # lag window used [h], <= pre_window_lag
    "output_states": [[0, "T_i"], [1, "T_h"]],
}
# Custom objective: weighted RMSE on (T_i, T_h) in objectives.py next to this notebook
# (the path is resolved relative to identification_config.json).
metric = "RMSE"
optimization_config = {"objective_function": "objectives.py:weighted_temp_objective", "pop_size": 100, "n_gen": 100}

selection_config  = {"criteria": "train error", "select_R_star": True, "threshold_config": {"type": "percentage", "x": 0.05}}
clustering_config = {"min_samples_ratio": 0.1, "cl_eval_metric": metric, "cl_eval_criteria": "train error"}

identification_config = {
    "model_config":        model_config,
    "optimization_config": optimization_config,
    "selection_config":    selection_config,
    "clustering_config":   clustering_config,
}
with open(OUT_DIR_PATH / "identification_config.json", "w") as f:
    json.dump(identification_config, f, indent=4)

## TRAIN (CLI)

No `--building-config`: the bounds of this model do not depend on the building.